## Sumativa 02– Análisis de datos Juegos olimpicos con Apache Spark


**MCDI502: Gestión de Datos y Tecnologías Big Data**  

**Proyecto:** Sumativa 02– Análisis de datos Juegos olimpicos con Apache Spark 
**Dataset:** Juegos olimpicos 
**Integrantes:** Enzo Pinilla, Claudio Alarcón y Luis Rodrigo Espinoza  
**Docente:** Eduardo Navarro Lorenzo 


## 1. Instalar entorno de Spark: configuración inicial

En esta sección se prepara el entorno necesario para utilizar Apache Spark. 

La instalación de Java y PySpark permite ejecutar el motor de procesamiento distribuido, mientras que la configuración de las variables de entorno permite que Spark encuentre correctamente Java, Python y sus propias librerías.

La creación y validación del entorno se realizará en el punto 1.1.

## 1.3. Crear la SparkSession con el nombre "OlimpiadasAnalysis" y el SparkContext.

In [2]:
# esto la creacion del entorno de spark y la creacion de la sesion de spark
## luego se listan los archivos disponibles en el directorio "Formativa" y se muestran las variables de entorno relacionadas con Spark y Java.

import os
import sys
import subprocess
from pathlib import Path

import pyspark
import findspark

java_home = subprocess.check_output(["/usr/libexec/java_home"]).decode().strip()
os.environ["JAVA_HOME"] = java_home
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)

findspark.init()

sc = pyspark.SparkContext.getOrCreate()
spark = (
    pyspark.sql.SparkSession.builder
    .master("local[*]")
    .appName("OlimpiadasAnalysis")
    .getOrCreate()
)

data_dir = Path.cwd() / "sumativa"

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("SPARK_HOME =", os.environ["SPARK_HOME"])
print("SparkSession creada:", spark.conf.get("spark.app.name"))
print("SparkContext activo:", sc)
print("Archivos disponibles en sumativa:")
for archivo in sorted(data_dir.iterdir()):
    print("-", archivo.name)

JAVA_HOME = /Library/Java/JavaVirtualMachines/jdk-18.0.1.1.jdk/Contents/Home
SPARK_HOME = /Users/sauriomac/Documents/spark-semana2/.venv/lib/python3.10/site-packages/pyspark
SparkSession creada: OlimpiadasAnalysis
SparkContext activo: <SparkContext master=local[*] appName=pyspark-shell>
Archivos disponibles en sumativa:
- deportista.csv
- deportista2.csv
- equipo.csv
- evento.csv
- juegos.json
- resultados.csv


## 2. RDDs: creación y unión (ID 3.1)



### 2.1 Creen un RDD llamado deportista con 6 particiones, leyendo el archivo deportista.csv.


In [3]:
# se crea un RDD a partir sdel archivo deportista.csv, dividiendo el archivo en 6 particiones.
deportista = sc.textFile("sumativa/deportista.csv", 6)

### 2.2 Creen un RDD llamado deportista2, leyendo el archivo deportista2.csv.


In [7]:
# se crea un RDD a partir sdel archivo deportista.csv, dividiendo el archivo en 6 particiones.
deportista2 = sc.textFile("sumativa/deportista.csv", 6)


sumativa/deportista.csv MapPartitionsRDD[7] at textFile at NativeMethodAccessorImpl.java:0


### 2.3 Creen un nuevo RDD llamado deportistaTotal que contenga la unión de los RDDs deportista y deportista2.


In [5]:
# se crea un RDD a partir del archivo deportistaTotal que contenga la union de los RDDS deportistas y deportida2.
deportistaTotal = deportista.union(deportista2)

### 2.4 Muestren la cantidad de registros contenidos en deportistaTotal.


In [6]:
# muestra la cantidad de registros
deportistaTotal.count()


135572

### 2.5 Conviertan el RDD deportistaTotal en un DataFrame llamado deportista con las siguientes columnas: "deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id".

In [8]:
## se conevierte el RDD jugadorTotal en un DataFrame llamado jugadores, utilizando la función Row para definir la estructura de los datos y especificando los nombres de las columnas.
## y luego se muestra el contenido del DataFrame jugadores utilizando el método show() para ver las primeras filas del DataFrame.
from pyspark.sql import Row


filas = deportistaTotal.map(lambda line: line.split(",")).filter(lambda fila: fila[0] != "deportista_id")

datos = filas.map(lambda fila: Row(
    deportista_id=fila[0],
    nombre=fila[1],
    genero=fila[2],
    edad=fila[3],
    altura=fila[4],
    peso=fila[5],
    equipo_id=fila[6],
))

deportista = datos.toDF([
    "deportista_id",
    "nombre",
    "genero",
    "edad",
    "altura",
    "peso",
    "equipo_id",
])

deportista.show()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|  80|      199|
|            2|            A Lamusi|     1|  23|   170|  60|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0|   0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0|   0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|  82|      705|
|            6|     Per Knut Aaland|     1|  31|   188|  75|     1096|
|            7|        John Aalberg|     1|  31|   183|  72|     1096|
|            8|Cornelia Cor Aalt...|     2|  18|   168|   0|      705|
|            9|    Antti Sami Aalto|     1|  26|   186|  96|      350|
|           10|Einar Ferdinand E...|     1|  26|     0|   0|      350|
|           11|  Jorma Ilmari Aalto|     1|  22|   182|76.5|      350|
|     

## 3. RDDs: transformaciones (ID 3.2)

### 3.1 Creen un RDD llamado MayorEdad que permita filtrar deportistas mayores de edad.


In [13]:
# crear un RDD  llamado MayorEdad que permite filtrar deportistas mayores de edad.

MayorEdad = deportista.filter(deportista.edad > 18)

In [14]:
MayorEdad.show()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|  80|      199|
|            2|            A Lamusi|     1|  23|   170|  60|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0|   0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0|   0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|  82|      705|
|            6|     Per Knut Aaland|     1|  31|   188|  75|     1096|
|            7|        John Aalberg|     1|  31|   183|  72|     1096|
|            9|    Antti Sami Aalto|     1|  26|   186|  96|      350|
|           10|Einar Ferdinand E...|     1|  26|     0|   0|      350|
|           11|  Jorma Ilmari Aalto|     1|  22|   182|76.5|      350|
|           12|   Jyri Tapani Aalto|     1|  31|   172|  70|      350|
|     

### 3.2 Creen un RDD llamado Deportistas_mujer que muestre solo los deportistas del género femenino (sexo=2).


In [19]:
# crear un RDD  llamado deportistas_mujer filtrar deportista femenino genero=2

Deportistas_mujer = deportista.filter(deportista.genero == "2")

In [20]:
Deportistas_mujer.show()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            5|Christine Jacoba ...|     2|  21|   185|  82|      705|
|            8|Cornelia Cor Aalt...|     2|  18|   168|   0|      705|
|           13|  Minna Maarit Aalto|     2|  30|   159|55.5|      350|
|           14|Pirjo Hannele Aal...|     2|  32|   171|  65|      350|
|           21|Ragnhild Margreth...|     2|  27|   163|   0|      742|
|           22|       Andreea Aanei|     2|  22|   170| 125|      861|
|           26|Agnes Erika Aanon...|     2|  17|   169|  65|      742|
|           29|Willemien Aardenburg|     2|  22|     0|   0|      705|
|           37|  Ann Kristin Aarnes|     2|  23|   182|  64|      742|
|           49|        Moonika Aava|     2|  24|   168|  65|      331|
|           65|   Patimat Abakarova|     2|  21|   165|  49|       74|
|     

### 3.3 Conviertan a mayúsculas todas las palabras del RDD deportistaTotal.

In [ ]:
# se convierte a mayusculas todas las palabras del RDD deportistaTotal

deportistaTotal_mayusculas = deportistaTotal.map(
    lambda linea: linea.upper()
)

deportistaTotal_mayusculas.collect()


## 4. DataFrames: creación e integración (ID 3.1)




### 4.1 Creen los DataFrames Evento, Resultado, Equipos y Juego a partir de los archivos eventos.csv, equipos.csv, resultado.csv y juego.json.


In [48]:
# Se define la ruta del archivo CSV que contiene los eventos.
Evento = "sumativa/evento.csv"

# Se define la ruta del archivo CSV que contiene los resultados.
Resultado = "sumativa/resultados.csv"

# Se define la ruta del archivo CSV que contiene los equipos.
Equipos = "sumativa/equipo.csv"

# Se define la ruta del archivo JSON que contiene los juegos.
Juego = "sumativa/juegos.json"


# Se crea el DataFrame Df_Evento leyendo el archivo eventos.csv.
Df_Evento = spark.read.csv(
    Evento,              # Ruta del archivo CSV.
    sep=",",             # Separador utilizado entre los campos.
    header=True,         # Indica que la primera fila contiene los nombres de las columnas.
    inferSchema=True     # Permite que Spark determine automáticamente los tipos de datos.
)

# Se crea el DataFrame Df_Resultado leyendo el archivo resultado.csv.
Df_Resultado = spark.read.csv(
    Resultado,           # Ruta del archivo CSV.
    sep=";",             # Separador utilizado entre los campos.
    header=True,         # Utiliza la primera fila como encabezado.
    inferSchema=True     # Infiere automáticamente los tipos de las columnas.
)

# Se crea el DataFrame Df_Equipos leyendo el archivo equipos.csv.
Df_Equipos = spark.read.csv(
    Equipos,             # Ruta del archivo CSV.
    sep=",",             # Separador utilizado entre los campos.
    header=True,         # Utiliza la primera fila como encabezado.
    inferSchema=True     # Infiere automáticamente los tipos de las columnas.
)

# Se crea el DataFrame Df_Juego leyendo el archivo juego.json.
Df_Juego = (
    spark.read
    .option("multiLine", "true")  # Permite leer un arreglo JSON distribuido en varias líneas.
    .json(Juego)                  # Lee el archivo JSON desde la ruta almacenada en Juego.
)


# Se muestran las primeras 10 filas del DataFrame de eventos.
Df_Evento.show(10, truncate=False)

# Se muestra el esquema y los tipos de datos del DataFrame de eventos.
Df_Evento.printSchema()


# Se muestran las primeras 10 filas del DataFrame de resultados.
Df_Resultado.show(10, truncate=False)

# Se muestra el esquema y los tipos de datos del DataFrame de resultados.
Df_Resultado.printSchema()


# Se muestran las primeras 10 filas del DataFrame de equipos.
Df_Equipos.show(10, truncate=False)

# Se muestra el esquema y los tipos de datos del DataFrame de equipos.
Df_Equipos.printSchema()


# Se muestran las primeras 10 filas del DataFrame de juegos.
Df_Juego.show(10, truncate=False)

# Se muestra el esquema y los tipos de datos del DataFrame de juegos.
Df_Juego.printSchema()

+----------+---------------------------------------------------+----------+
|cod_evento|evento                                             |deporte_id|
+----------+---------------------------------------------------+----------+
|1         |Basketball Men's Basketball                        |1         |
|2         |Judo Men's Extra-Lightweight                       |2         |
|3         |Football Men's Football                            |3         |
|4         |Tug-Of-War Men's Tug-Of-War                        |4         |
|5         |Speed Skating Women's 500 metres                   |5         |
|6         |Speed Skating Women's 1000 metres                  |5         |
|7         |Cross Country Skiing Men's 10 kilometres           |6         |
|8         |Cross Country Skiing Men's 50 kilometres           |6         |
|9         |Cross Country Skiing Men's 10/15 kilometres Pursuit|6         |
|10        |Cross Country Skiing Men's 4 x 10 kilometres Relay |6         |
+----------+

### 4.2 Realicen la optimización de los DataFrames creados.


In [45]:
# Se almacena el DataFrame en memoria caché para acelerar

Df_Evento.cache()
Df_Equipos.cache()
Df_Juego.cache()

# Se persiste el DataFrame para conservar sus particiones

deportista.persist()
deportistaTotal.persist()
Df_Resultado.persist()

26/07/26 20:33:07 WARN CacheManager: Asked to cache already cached data.
26/07/26 20:33:07 WARN CacheManager: Asked to cache already cached data.
26/07/26 20:33:07 WARN CacheManager: Asked to cache already cached data.
26/07/26 20:33:07 WARN CacheManager: Asked to cache already cached data.
26/07/26 20:33:08 WARN CacheManager: Asked to cache already cached data.


DataFrame[resultado_id;medalla;deportista_id;juego_id;evento_id: string]

### 4.3 Crearen un nuevo DataFrame que contenga los datos de todos los DataFrames (deportista, evento, equipo, resultado y juego).

Relaciones utilizadas:

- `Df_Resultado.deportista_id = deportista.deportista_id`
- `deportista.equipo_id = Equipos.id`
- `Df_Resultado.evento_id = Df_Evento.cod_evento`
- `Df_Resultado.juego_id = Df_Juego.juego_id`


In [49]:
# Se crean alias para identificar claramente las columnas durante los joins.

r = Df_Resultado.alias("r")
d = deportista.alias("d")
e = Df_Equipos.alias("e")
ev = Df_Evento.alias("ev")
j = Df_Juego.alias("j")

olimpiadas_completo = (
    r
    .join(
        d,
        r["deportista_id"] == d["deportista_id"],
        "left"
    )
    .join(
        e,
        d["equipo_id"] == e["id"],
        "left"
    )
    .join(
        ev,
        r["evento_id"] == ev["cod_evento"],
        "left"
    )
    .join(
        j,
        r["juego_id"] == j["juego_id"],
        "left"
    )
    .select(
        # Resultado
        r["resultado_id"],
        r["medalla"],

        # Deportista
        d["deportista_id"],
        d["nombre"],
        d["genero"],
        d["edad"],
        d["altura"],
        d["peso"],
        d["equipo_id"],

        # Equipo
        e["equipo"],
        e["sigla"],

        # Evento
        ev["cod_evento"].alias("evento_id"),
        ev["evento"],
        ev["deporte_id"],

        # Juego
        j["juego_id"],
        j["ano"],
        j["temporada"],
        j["ciudad"],
    )
)

olimpiadas_completo.show(20, truncate=False)

+------------+-------+-------------+-------------------+------+----+------+----+---------+-------------+-----+---------+---------------------------------------------------+----------+--------+-------------+---------+--------+
|resultado_id|medalla|deportista_id|nombre             |genero|edad|altura|peso|equipo_id|equipo       |sigla|evento_id|evento                                             |deporte_id|juego_id|ano          |temporada|ciudad  |
+------------+-------+-------------+-------------------+------+----+------+----+---------+-------------+-----+---------+---------------------------------------------------+----------+--------+-------------+---------+--------+
|1           |NA     |1            |A Dijiang          |1     |24  |180   |80  |199      |China        |CHN  |1        |Basketball Men's Basketball                        |1         |39      |1992 Verano  |1992     |Verano  |
|1           |NA     |1            |A Dijiang          |1     |24  |180   |80  |199      |China 

### 5. Paralelismo (ID 3.4) Paralelicen el DataFrame resultante a 5 particiones.


In [ ]:
# Se redistribuye el DataFrame completo en 5 particiones.

olimpiadas_completo = olimpiadas_completo.repartition(5)

# Se comprueba la nueva cantidad de particiones.

print(
    "Particiones: ",
    olimpiadas_completo.rdd.getNumPartitions()
)

Particiones:  5


6. Inspección del DataFrame (ID 3.1)

Muestren la cantidad de filas, el tipo de datos y el esquema del DataFrame.



In [53]:
# mostrar la cantidade de filas, el tipo de datos y el esquema del dataframe mundial_completo utilizando los métodos count(), dtypes y printSchema() respectivamente. Esto permite obtener información sobre la cantidad de registros, los tipos de datos de cada columna y la estructura del DataFrame mundial_completo.
print("Cantidad de filas en el DataFrame mundial_completo:", olimpiadas_completo.count())

tipo_de_col = olimpiadas_completo.dtypes

print(
    "Tipos de datos de cada columna:\n" +
    "\n".join(
        f"- {columna}: {tipo}"
        for columna, tipo in olimpiadas_completo.dtypes
    )
)
print("=========================================================")

print("Esquema del DataFrame mundial_completo:")
olimpiadas_completo.printSchema()

Cantidad de filas en el DataFrame mundial_completo: 405836
Tipos de datos de cada columna:
- resultado_id: int
- medalla: string
- deportista_id: string
- nombre: string
- genero: string
- edad: string
- altura: string
- peso: string
- equipo_id: string
- equipo: string
- sigla: string
- evento_id: string
- evento: string
- deporte_id: string
- juego_id: bigint
- ano: string
- temporada: bigint
- ciudad: string
Esquema del DataFrame mundial_completo:
root
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- deportista_id: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- edad: string (nullable = true)
 |-- altura: string (nullable = true)
 |-- peso: string (nullable = true)
 |-- equipo_id: string (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: string (nullable = true)
 |-- evento_id: string (nullable = true)
 |-- evento: string (nullable = true)
 |-- deporte_id: string (nulla

### 7. Columnas calculadas


### 7.1 Creen una columna calculada llamada IMC que determine el Índice de Masa Corporal de cada deportista.

In [57]:
# Se calcula el Índice de Masa Corporal:
# IMC = peso / (altura en metros)²
#
# Los valores 0 de peso o altura representan datos no informados.
# En esos casos el IMC queda como null para evitar una división por cero.
from pyspark.sql.functions import round as spark_round, when, col

olimpiadas_calculado = olimpiadas_completo.withColumn(
    "IMC",
    when(
        (col("altura") > 0) & (col("peso") > 0),
        spark_round(
            col("peso").cast("double")
            / ((col("altura").cast("double") / 100) ** 2),
            2
        )
    ).otherwise(None)
)

olimpiadas_calculado.select(
    "deportista_id",
    "nombre",
    "altura",
    "peso",
    "IMC"
).show(20, truncate=False)

+-------------+---------------------------+------+----+-----+
|deportista_id|nombre                     |altura|peso|IMC  |
+-------------+---------------------------+------+----+-----+
|41749        |Marin Gopov                |181   |72  |21.98|
|44           |Meelis Aasme               |182   |73  |22.04|
|44           |Meelis Aasme               |182   |73  |22.04|
|67318        |Dennis Lawrence            |191   |79  |21.66|
|18997        |Eduardo Castro Almanza     |173   |63  |21.05|
|50           |Arvi Aavik                 |185   |106 |30.97|
|31693        |G. Eichhorn                |0     |0   |NULL |
|28753        |John Ivor Disley           |180   |71  |21.91|
|67276        |Lisiate Lavulo             |0     |64  |NULL |
|22944        |Edmund Kirkland Kirk Cooper|183   |77  |22.99|
|61613        |Bernd Knuppel Barenberg    |183   |96  |28.67|
|24304        |Diane S. Cummins           |165   |50  |18.37|
|39667        |Giulio Gerardi             |0     |0   |NULL |
|51989  

### 7.2 Creen una columna calculada llamada Descripción_sexo que determine el sexo de los deportistas (sexo=1 es hombre, sexo=2 es mujer).


In [59]:
# Se crea la descripción del sexo a partir de la columna genero
# genero = 1: Hombre
# genero = 2: Mujer

olimpiadas_calculado = olimpiadas_calculado.withColumn(
    "Descripción_sexo",
    when(col("genero") == 1, "Hombre")
    .when(col("genero") == 2, "Mujer")
    .otherwise("Sin información")
)

olimpiadas_calculado.select(
    "deportista_id",
    "nombre",
    "genero",
    "Descripción_sexo"
).show(60, truncate=False)

+-------------+----------------------------------+------+----------------+
|deportista_id|nombre                            |genero|Descripción_sexo|
+-------------+----------------------------------+------+----------------+
|62701        |Wolfram Koppen                    |1     |Hombre          |
|42834        |Manfred Grieshofer                |1     |Hombre          |
|65630        |Philip Laats                      |1     |Hombre          |
|42248        |Stefanie Maria Steffi Graf        |2     |Mujer           |
|42248        |Stefanie Maria Steffi Graf        |2     |Mujer           |
|59929        |Kim HaYun                         |1     |Hombre          |
|28891        |Kazimierz Dugopolski              |1     |Hombre          |
|66263        |Abdel Majid Lamris                |1     |Hombre          |
|38002        |Giuseppe Galante                  |1     |Hombre          |
|57174        |Georg Kandlinger                  |1     |Hombre          |
|57174        |Georg Kand

## 8. Agregaciones requeridas (ID 3.3)



### 8.1 Creen un DataFrame que muestre la cantidad de medallas (oro, bronce, plata) obtenidas por cada Equipo.


In [60]:
# Se filtran solamente las medallas válidas.
# Después se utiliza pivot para crear una columna por cada tipo de medalla.

from pyspark.sql.functions import (
    avg,
    coalesce,
    count,
    desc,
    lit,
    max as spark_max,
    min as spark_min,
    sum as spark_sum,
)

medallas_equipo = (
    olimpiadas_calculado
    .filter(col("medalla").isin("Gold", "Silver", "Bronze"))
    .withColumn(
        "equipo",
        coalesce(col("equipo"), lit("Sin equipo registrado"))
    )
    .groupBy("equipo")
    .pivot("medalla", ["Gold", "Silver", "Bronze"])
    .agg(count("*"))
    .fillna(0)
    .withColumnRenamed("Gold", "Oro")
    .withColumnRenamed("Silver", "Plata")
    .withColumnRenamed("Bronze", "Bronce")
    .orderBy(
        desc("Oro"),
        desc("Plata"),
        desc("Bronce")
    )
)

medallas_equipo.show(100, truncate=False)

+---------------------------------------------------+----+-----+------+
|equipo                                             |Oro |Plata|Bronce|
+---------------------------------------------------+----+-----+------+
|Sin equipo registrado                              |6801|6641 |6768  |
|United States                                      |2582|1634 |1290  |
|Soviet Union                                       |1004|704  |712   |
|Germany                                            |628 |636  |674   |
|France                                             |558 |614  |674   |
|Great Britain                                      |544 |634  |592   |
|Italy                                              |512 |492  |504   |
|Hungary                                            |496 |352  |370   |
|Canada                                             |446 |408  |434   |
|Sweden                                             |418 |498  |552   |
|East Germany                                       |382 |274  |

### 8.2 Creen un DataFrame que obtenga la suma, promedio, máximo y mínimo de la edad de los deportistas, agrupado por el tipo de medalla obtenida.


In [61]:
# Las edades con valor 0 se excluyen porque representan datos no informados.

edad_por_medalla = (
    olimpiadas_calculado
    .filter(
        col("medalla").isin("Gold", "Silver", "Bronze")
        & (col("edad") > 0)
    )
    .groupBy("medalla")
    .agg(
        spark_sum("edad").alias("suma_edad"),
        spark_round(avg("edad"), 2).alias("promedio_edad"),
        spark_max("edad").alias("edad_maxima"),
        spark_min("edad").alias("edad_minima"),
    )
    .orderBy("medalla")
)

edad_por_medalla.show(truncate=False)

+-------+---------+-------------+-----------+-----------+
|medalla|suma_edad|promedio_edad|edad_maxima|edad_minima|
+-------+---------+-------------+-----------+-----------+
|Bronze |302888.0 |23.68        |72         |13         |
|Gold   |307174.0 |23.61        |64         |11         |
|Silver |299940.0 |23.71        |73         |11         |
+-------+---------+-------------+-----------+-----------+



### 8.3 Creen un nuevo DataFrame llamado temporada que obtenga la suma, promedio, máximo y mínimo de las alturas de los deportistas, agrupado por temporada.


In [62]:
# Las alturas con valor 0 se excluyen porque representan datos no informados.

temporada = (
    olimpiadas_calculado
    .filter(
        col("temporada").isNotNull()
        & (col("altura") > 0)
    )
    .groupBy("temporada")
    .agg(
        spark_sum("altura").alias("suma_altura"),
        spark_round(avg("altura"), 2).alias("promedio_altura"),
        spark_max("altura").alias("altura_maxima"),
        spark_min("altura").alias("altura_minima"),
    )
    .orderBy("temporada")
)

temporada.show(100, truncate=False)

+---------+-----------+---------------+-------------+-------------+
|temporada|suma_altura|promedio_altura|altura_maxima|altura_minima|
+---------+-----------+---------------+-------------+-------------+
|1896     |12076.0    |177.59         |188          |154          |
|1900     |15860.0    |176.22         |188          |153          |
|1904     |40582.0    |174.92         |189          |155          |
|1906     |37324.0    |177.73         |195          |167          |
|1908     |87056.0    |176.94         |201          |160          |
|1912     |130388.0   |176.2          |200          |157          |
|1920     |128816.0   |175.5          |197          |156          |
|1924     |167990.0   |174.99         |200          |155          |
|1928     |160626.0   |175.74         |211          |155          |
|1932     |214252.0   |174.47         |200          |147          |
|1936     |194650.0   |175.99         |204          |147          |
|1948     |218136.0   |175.92         |213      

### 8.4 Creen un nuevo DataFrame llamado sexo que obtenga la suma, promedio, máximo y mínimo de la edad de los deportistas, agrupado por sexo.

In [63]:
# Se agrupa por la columna calculada Descripción_sexo.
# Las edades con valor 0 se excluyen porque representan datos no informados.

sexo = (
    olimpiadas_calculado
    .filter(col("edad") > 0)
    .groupBy("Descripción_sexo")
    .agg(
        spark_sum("edad").alias("suma_edad"),
        spark_round(avg("edad"), 2).alias("promedio_edad"),
        spark_max("edad").alias("edad_maxima"),
        spark_min("edad").alias("edad_minima"),
    )
    .orderBy("Descripción_sexo")
)

sexo.show(truncate=False)

+----------------+---------+-------------+-----------+-----------+
|Descripción_sexo|suma_edad|promedio_edad|edad_maxima|edad_minima|
+----------------+---------+-------------+-----------+-----------+
|Hombre          |4619248.0|24.44        |96         |11         |
|Mujer           |1554768.0|21.9         |72         |11         |
+----------------+---------+-------------+-----------+-----------+



# Bibliografía


- Maldonado, Sebastián (2022), Analytics y big data: ciencia de los datos aplicada al mundo de los negocios.
https://unab.primo.exlibrisgroup.com/permalink/56UAB_INST/1ebbirc/cdi_askewsholts_vlebooks_9788418982637Links.

- López Fandiño, V. M. (2023). Sistemas de Big Data. Alfaomega Grupo Editor.
https://unab.primo.exlibrisgroup.com/permalink/56UAB_INST/1ebbirc/cdi_elibro_books_ELB235054Links.

